# Linear-Probe Causal Intervention: An Alternative to MCQ Generation Accuracy

The MCQ pointing-game framework (`mcq_causal_intervention.ipynb`) measures the causal
effect of Vis-Head steering by **generating text** and parsing out a letter (A/B/C/D),
then comparing accuracy between baseline and steered conditions:
`ATE = P(correct | steered) - P(correct | baseline)`.

This notebook replaces the generation step with a **linear probe** on the model's
internal representation: at the final prompt token (right before generation would
start), extract the last transformer layer's hidden state, and train a logistic
regression classifier to predict the correct option letter directly from that vector
— separately for the baseline condition and the steered condition. This asks a
different but related question: does the intervention change what the model's
*internal representation* encodes about the answer, independent of whether it can
also verbalize that answer correctly under the MCQ generation format (which we know
from earlier notebooks is itself imperfect — e.g. "answer D" biases, weak
instruction-following in smaller models)?

**Probe ATE** = `probe_accuracy(steered) - probe_accuracy(baseline)`, with the same
sign/interpretation as the generation-based ATE. We also report macro one-vs-rest
ROC-AUC (accuracy alone can be misleading with a 4-way near-random classifier), and
compare directly against generation-based MCQ accuracy on the *same* samples as a
consistency check between the two causal-effect measures.

In [1]:
%matplotlib inline
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "vis_head").exists():
    raise RuntimeError("Run this notebook from the repository root (vis-head/).")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import re
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import label_binarize

from vis_head.common import DEFAULT_MODEL_ID, DEFAULT_SEED
from vis_head.vir import aggregate_region_attention, collect_last_query_attentions, rank_heads_by_score
from vis_head.imagenet_grid import DEFAULT_IMAGENET_ROOT, PROMPT_TEMPLATES, mcq_prompt, list_val_class_dirs, load_class_names, sample_grid
from vis_head.modeling import decode_generated_text, find_image_token_range, load_model_and_processor, model_dims, prepare_inputs, run_generation
from vis_head.regions import assign_grid_cells_to_tokens, region_positions_from_ids
from vis_head.steering import group_heads_by_layer, intervention_positions, make_static_attention_mask_hook, register_mask_hooks, remove_handles

DEVICE = "cuda:0"
SEED = DEFAULT_SEED
ROWS, COLS = 2, 2
N_CELLS = ROWS * COLS
CELL_SIZE = 256
N_DISCOVERY_SAMPLES = 300
TOP_K_HEADS = 15
N_PROBE_SAMPLES = 400   # needs enough per class (4-way) for a reasonable probe fit
N_OPTIONS = 4
OPTION_LETTERS = ["A", "B", "C", "D"][:N_OPTIONS]

imagenet_class_dirs = list_val_class_dirs(DEFAULT_IMAGENET_ROOT)
imagenet_class_names = load_class_names(DEFAULT_IMAGENET_ROOT)
print(f"Model: {DEFAULT_MODEL_ID}")

Model: Qwen/Qwen3-VL-8B-Instruct


In [2]:
model, processor = load_model_and_processor(model_id=DEFAULT_MODEL_ID, device=DEVICE)
n_layers, n_heads, spatial_merge = model_dims(model)
print(f"{n_layers} layers x {n_heads} heads")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

36 layers x 32 heads


## Discovery: rank Vis-Heads (same as other notebooks, `find` phrasing)

In [3]:
rng = np.random.RandomState(SEED)
raw_sum = np.zeros((n_layers, n_heads), dtype=np.float64)
valid = 0
for _ in range(N_DISCOVERY_SAMPLES):
    grid = sample_grid(rows=ROWS, cols=COLS, cell_size=CELL_SIZE, rng=rng,
                        class_dirs=imagenet_class_dirs, class_names=imagenet_class_names)
    target_cell = int(rng.randint(N_CELLS))
    prompt = PROMPT_TEMPLATES["find"].format(name=grid.cell_names[target_cell])
    try:
        inputs = prepare_inputs(processor, grid.grid, prompt, DEVICE)
        region_ids, _ = assign_grid_cells_to_tokens(image_grid_thw=inputs["image_grid_thw"], rows=ROWS, cols=COLS, spatial_merge=spatial_merge)
        attn = collect_last_query_attentions(model, inputs)
        region_attn = aggregate_region_attention(attn_at_query=attn, inputs=inputs, processor=processor, region_ids=region_ids, n_regions=N_CELLS)
        raw_sum += region_attn[:, :, target_cell]
        valid += 1
    except Exception as exc:
        print(f"Skipping grid: {exc}")

print(f"Discovery: valid={valid}/{N_DISCOVERY_SAMPLES}")
vis_head_scores = (raw_sum / max(valid, 1)).astype(np.float32)
ranked = rank_heads_by_score(vis_head_scores)
heads_by_layer = group_heads_by_layer([(r["layer"], r["head"]) for r in ranked[:TOP_K_HEADS]])
print(f"Top-{TOP_K_HEADS} heads:", [(r["layer"], r["head"]) for r in ranked[:TOP_K_HEADS]])

Discovery: valid=300/300
Top-15 heads: [(21, 11), (24, 29), (20, 15), (0, 27), (21, 24), (2, 17), (0, 5), (0, 8), (0, 29), (23, 28), (5, 29), (21, 18), (21, 8), (21, 10), (9, 14)]


## Build MCQ samples (no-cue prompt, same design as `mcq_causal_intervention.ipynb`)

In [4]:
def build_mcq_sample(rng):
    grid = sample_grid(rows=ROWS, cols=COLS, cell_size=CELL_SIZE, rng=rng,
                        class_dirs=imagenet_class_dirs, class_names=imagenet_class_names)
    target_cell = int(rng.randint(N_CELLS))
    correct_name = grid.cell_names[target_cell]
    other_names = [n for i, n in enumerate(grid.cell_names) if i != target_cell]
    n_distractors = min(N_OPTIONS - 1, len(other_names))
    distractor_idx = rng.choice(len(other_names), size=n_distractors, replace=False)
    distractors = [other_names[i] for i in distractor_idx]
    options = [correct_name] + distractors
    order = rng.permutation(len(options))
    options = [options[i] for i in order]
    correct_letter = OPTION_LETTERS[int(np.where(order == 0)[0][0])]
    option_lines = "  ".join(f"{l}) {n}" for l, n in zip(OPTION_LETTERS[:len(options)], options))
    prompt = f"Which of the following is shown in this image? {option_lines}. Answer with only the letter."
    location_prompt = mcq_prompt(target_cell + 1, ROWS, COLS, options, OPTION_LETTERS[:len(options)])
    return {"grid": grid, "target_cell": target_cell, "options": options,
            "correct_letter": correct_letter, "prompt": prompt, "location_prompt": location_prompt}


def extract_letter(text, valid_letters):
    match = re.search(r"\b([" + "".join(valid_letters) + r"])\b", text.upper())
    return match.group(1) if match else None


probe_rng = np.random.RandomState(SEED + 555)
probe_samples = [build_mcq_sample(probe_rng) for _ in range(N_PROBE_SAMPLES)]
print(f"Built {len(probe_samples)} MCQ samples")

Built 400 MCQ samples


## Extract final-token, last-layer hidden states + generation-based MCQ accuracy

For each sample and each condition (baseline / steered), one forward pass with
`output_hidden_states=True` gives the probe input (no generation needed for the
probe); a second short `run_generation` call (reusing the same hooks) gives the
letter-accuracy comparison point.

In [5]:
def get_final_hidden_state(inputs):
    """Last transformer layer's hidden state at the final prompt token —
    the representation immediately before the model would start generating."""
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True, use_cache=False)
    last_layer = out.hidden_states[-1]          # (batch, seq_len, hidden)
    return last_layer[0, -1, :].float().cpu().numpy()


def run_condition(sample, heads_by_layer_or_none, prompt_key="prompt"):
    inputs = prepare_inputs(processor, sample["grid"].grid, sample[prompt_key], DEVICE)
    prompt_length = int(inputs["input_ids"].shape[1])
    handles = []
    if heads_by_layer_or_none is not None:
        img_start, img_end = find_image_token_range(inputs, processor)
        region_ids, _ = assign_grid_cells_to_tokens(image_grid_thw=inputs["image_grid_thw"], rows=ROWS, cols=COLS, spatial_merge=spatial_merge)
        positions = region_positions_from_ids(img_start=img_start, region_ids=region_ids, n_regions=N_CELLS)
        target_positions = positions[sample["target_cell"]]
        other_positions = [p for i in range(N_CELLS) if i != sample["target_cell"] for p in positions[i]]
        suppress_positions, boost_positions, pad = intervention_positions(
            mode="boost_suppress", target_positions=target_positions, other_image_positions=other_positions,
            img_start=img_start, img_end=img_end, prompt_length=prompt_length)
        hook_by_layer = {
            l: make_static_attention_mask_hook(head_indices=hh, suppress_positions=suppress_positions,
                                                boost_positions=boost_positions, n_query_heads=n_heads,
                                                device=DEVICE, decode_only=False, pad_with_suppress=pad)
            for l, hh in heads_by_layer_or_none.items()
        }
        handles = register_mask_hooks(model, hook_by_layer)
    try:
        hidden = get_final_hidden_state(inputs)
        seq = run_generation(model=model, inputs=inputs, max_new_tokens=6)
    finally:
        remove_handles(handles)
    text = decode_generated_text(processor, seq, prompt_length)
    predicted = extract_letter(text, OPTION_LETTERS[:len(sample["options"])])
    gen_correct = predicted == sample["correct_letter"]
    return hidden, gen_correct


baseline_hidden, baseline_gen_correct = [], []
location_hidden, location_gen_correct = [], []
steered_hidden, steered_gen_correct = [], []
labels = []

from tqdm.auto import tqdm
for sample in tqdm(probe_samples, desc="Extracting hidden states + generation"):
    h_b, c_b = run_condition(sample, None, "prompt")
    h_l, c_l = run_condition(sample, None, "location_prompt")
    h_s, c_s = run_condition(sample, heads_by_layer, "prompt")
    baseline_hidden.append(h_b); baseline_gen_correct.append(c_b)
    location_hidden.append(h_l); location_gen_correct.append(c_l)
    steered_hidden.append(h_s); steered_gen_correct.append(c_s)
    labels.append(sample["correct_letter"])

baseline_hidden = np.stack(baseline_hidden)
location_hidden = np.stack(location_hidden)
steered_hidden = np.stack(steered_hidden)
labels = np.array(labels)
print(f"Hidden state shape: {baseline_hidden.shape}")
print(f"Class balance: {dict(zip(*np.unique(labels, return_counts=True)))}")
print(f"Generation accuracy -- baseline: {np.mean(baseline_gen_correct):.3f}  "
      f"location-cue: {np.mean(location_gen_correct):.3f}  steered: {np.mean(steered_gen_correct):.3f}")

Extracting hidden states + generation:   0%|          | 0/400 [00:00<?, ?it/s]

Hidden state shape: (400, 4096)
Class balance: {np.str_('A'): np.int64(107), np.str_('B'): np.int64(108), np.str_('C'): np.int64(111), np.str_('D'): np.int64(74)}
Generation accuracy -- baseline: 0.260  location-cue: 0.988  steered: 0.637


## Train linear probes (baseline hidden states vs. steered hidden states)

Multinomial logistic regression, held-out test split, macro one-vs-rest ROC-AUC
(chance = 0.25 accuracy, 0.5 AUC for 4-way).

In [6]:
def fit_and_eval_probe(X, y, seed=SEED):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
    clf = LogisticRegression(max_iter=2000, multi_class="multinomial", C=1.0)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    y_proba = clf.predict_proba(X_test)
    y_test_bin = label_binarize(y_test, classes=clf.classes_)
    auc = roc_auc_score(y_test_bin, y_proba, average="macro", multi_class="ovr")
    return {"accuracy": acc, "auc": auc, "n_train": len(y_train), "n_test": len(y_test), "classes": list(clf.classes_)}


baseline_probe = fit_and_eval_probe(baseline_hidden, labels)
location_probe = fit_and_eval_probe(location_hidden, labels)
steered_probe = fit_and_eval_probe(steered_hidden, labels)

b_acc, b_auc = baseline_probe["accuracy"], baseline_probe["auc"]
loc_acc, loc_auc = location_probe["accuracy"], location_probe["auc"]
s_acc, s_auc = steered_probe["accuracy"], steered_probe["auc"]
print(f"Baseline probe:     accuracy={b_acc:.3f}  AUC={b_auc:.3f}  "
      f"(n_train={baseline_probe['n_train']}, n_test={baseline_probe['n_test']})")
print(f"Location-cue probe: accuracy={loc_acc:.3f}  AUC={loc_auc:.3f}")
print(f"Steered probe:      accuracy={s_acc:.3f}  AUC={s_auc:.3f}")
print(f"\nProbe ATE steered-vs-baseline  (accuracy): {s_acc - b_acc:+.3f}   (AUC): {s_auc - b_auc:+.3f}")
print(f"Probe ATE location-vs-baseline (accuracy): {loc_acc - b_acc:+.3f}   (AUC): {loc_auc - b_auc:+.3f}")
print(f"Probe delta steered-vs-location (accuracy): {s_acc - loc_acc:+.3f}   (AUC): {s_auc - loc_auc:+.3f}")
print(f"\nGeneration ATE (accuracy, same samples) -- steered-vs-baseline: {np.mean(steered_gen_correct) - np.mean(baseline_gen_correct):+.3f}  "
      f"location-vs-baseline: {np.mean(location_gen_correct) - np.mean(baseline_gen_correct):+.3f}")

/mnt/abka03/.conda/virheads/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


/mnt/abka03/.conda/virheads/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


/mnt/abka03/.conda/virheads/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Baseline probe:     accuracy=0.233  AUC=0.479  (n_train=280, n_test=120)
Location-cue probe: accuracy=0.983  AUC=0.999
Steered probe:      accuracy=0.633  AUC=0.867

Probe ATE steered-vs-baseline  (accuracy): +0.400   (AUC): +0.388
Probe ATE location-vs-baseline (accuracy): +0.750   (AUC): +0.520
Probe delta steered-vs-location (accuracy): -0.350   (AUC): -0.132

Generation ATE (accuracy, same samples) -- steered-vs-baseline: +0.377  location-vs-baseline: +0.728


## Cross-validated robustness check

A single 70/30 split can be noisy; repeat with multiple random splits to get a
distribution of the probe ATE rather than a single point estimate.

In [7]:
N_REPEATS = 20
baseline_accs, location_accs, steered_accs = [], [], []
baseline_aucs, location_aucs, steered_aucs = [], [], []
for i in range(N_REPEATS):
    bp = fit_and_eval_probe(baseline_hidden, labels, seed=SEED + i)
    lp = fit_and_eval_probe(location_hidden, labels, seed=SEED + i)
    sp = fit_and_eval_probe(steered_hidden, labels, seed=SEED + i)
    baseline_accs.append(bp["accuracy"]); location_accs.append(lp["accuracy"]); steered_accs.append(sp["accuracy"])
    baseline_aucs.append(bp["auc"]); location_aucs.append(lp["auc"]); steered_aucs.append(sp["auc"])

from scipy import stats as sstats
acc_delta_sb = np.array(steered_accs) - np.array(baseline_accs)
auc_delta_sb = np.array(steered_aucs) - np.array(baseline_aucs)
acc_delta_lb = np.array(location_accs) - np.array(baseline_accs)
auc_delta_lb = np.array(location_aucs) - np.array(baseline_aucs)
acc_delta_sl = np.array(steered_accs) - np.array(location_accs)
auc_delta_sl = np.array(steered_aucs) - np.array(location_aucs)

t_acc_sb, p_acc_sb = sstats.ttest_1samp(acc_delta_sb, 0.0)
t_auc_sb, p_auc_sb = sstats.ttest_1samp(auc_delta_sb, 0.0)
t_acc_lb, p_acc_lb = sstats.ttest_1samp(acc_delta_lb, 0.0)
t_auc_lb, p_auc_lb = sstats.ttest_1samp(auc_delta_lb, 0.0)
t_acc_sl, p_acc_sl = sstats.ttest_1samp(acc_delta_sl, 0.0)
t_auc_sl, p_auc_sl = sstats.ttest_1samp(auc_delta_sl, 0.0)

summary = pd.DataFrame([{
    "model": DEFAULT_MODEL_ID, "n_samples": N_PROBE_SAMPLES, "top_k_heads": TOP_K_HEADS,
    "baseline_probe_acc_mean": np.mean(baseline_accs), "location_probe_acc_mean": np.mean(location_accs),
    "steered_probe_acc_mean": np.mean(steered_accs),
    "probe_acc_ate_steer_vs_base": acc_delta_sb.mean(), "p_acc_steer_vs_base": p_acc_sb,
    "probe_acc_ate_loc_vs_base": acc_delta_lb.mean(), "p_acc_loc_vs_base": p_acc_lb,
    "probe_acc_delta_steer_vs_loc": acc_delta_sl.mean(), "p_acc_steer_vs_loc": p_acc_sl,
    "baseline_probe_auc_mean": np.mean(baseline_aucs), "location_probe_auc_mean": np.mean(location_aucs),
    "steered_probe_auc_mean": np.mean(steered_aucs),
    "probe_auc_ate_steer_vs_base": auc_delta_sb.mean(), "p_auc_steer_vs_base": p_auc_sb,
    "probe_auc_ate_loc_vs_base": auc_delta_lb.mean(), "p_auc_loc_vs_base": p_auc_lb,
    "probe_auc_delta_steer_vs_loc": auc_delta_sl.mean(), "p_auc_steer_vs_loc": p_auc_sl,
    "baseline_gen_acc": np.mean(baseline_gen_correct), "location_gen_acc": np.mean(location_gen_correct),
    "steered_gen_acc": np.mean(steered_gen_correct),
    "gen_acc_ate_steer_vs_base": np.mean(steered_gen_correct) - np.mean(baseline_gen_correct),
    "gen_acc_ate_loc_vs_base": np.mean(location_gen_correct) - np.mean(baseline_gen_correct),
}])
pd.set_option("display.width", 160)
print(summary.T.to_string())

/mnt/abka03/.conda/virheads/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


/mnt/abka03/.conda/virheads/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


/mnt/abka03/.conda/virheads/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


/mnt/abka03/.conda/virheads/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


/mnt/abka03/.conda/virheads/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


...

/mnt/abka03/.conda/virheads/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


/mnt/abka03/.conda/virheads/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


/mnt/abka03/.conda/virheads/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


/mnt/abka03/.conda/virheads/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


                                                      0
model                         Qwen/Qwen3-VL-8B-Instruct
n_samples                                           400
top_k_heads                                          15
baseline_probe_acc_mean                            0.23
location_probe_acc_mean                        0.979583
steered_probe_acc_mean                            0.625
probe_acc_ate_steer_vs_base                       0.395
p_acc_steer_vs_base                                 0.0
probe_acc_ate_loc_vs_base                      0.749583
p_acc_loc_vs_base                                   0.0
probe_acc_delta_steer_vs_loc                  -0.354583
p_acc_steer_vs_loc                                  0.0
baseline_probe_auc_mean                        0.465862
location_probe_auc_mean                        0.999401
steered_probe_auc_mean                         0.841134
probe_auc_ate_steer_vs_base                    0.375271
p_auc_steer_vs_base                             

## Result

(filled in after running)